# Scan QR codes and barcodes in Python

This notebook installs the published package
[`qrcode-barcode-scanner`](https://pypi.org/project/qrcode-barcode-scanner/0.11.0/) 0.11.0
and scans sample images with it. The scanner is a local Rust engine: every image is decoded in
this session and none is sent anywhere.

Run the cells in order, or choose **Run All**. Each cell checks its own result, so a cell that
finishes without an error did what it says.

QR Code AI web links · [QR Code AI](https://qrcode-ai.com/) ·
[QR code scanner](https://qrcode-ai.com/en-US/qr-code-scanner) ·
[Barcode scanner](https://qrcode-ai.com/en-US/barcode-scanner).

## 1. Install the package

The package has no dependency. On a host that prepared the session from this repository, such as
Binder, it is already installed and this cell changes nothing.

In [ ]:
%pip install --quiet qrcode-barcode-scanner==0.11.0

## 2. Check what is installed

The cell prints the Python version, the platform and the package version. It then compares the
SHA-256 of the installed native module with the modules of the wheels published on PyPI for this
version.

In [ ]:
import hashlib
import importlib.metadata
import pathlib
import platform

import qrcode_barcode_scanner as scanner

VERSION = "0.11.0"
# SHA-256 of the native module inside each wheel published on PyPI for this version.
PUBLISHED_MODULES = {
    "326431ff9f8402df257c38bb2d26037ae03e73bb6a0e7a381c4f3c4c9ab5ed09":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-macosx_10_12_x86_64.whl",
    "d1975a7b933c7440b4bd45295946e9a082286a0d6d18df7bdfdfc53fdceffcc7":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-macosx_11_0_arm64.whl",
    "3e353686899d4658f5b5560392ca4ef377789b83a0ee4fa13b36eaa6f349145d":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-manylinux_2_17_aarch64.manylinux2014_aarch64.whl",
    "de30ac77aee57c90ec4552af10e075fbd83aa1f775b53031cf5acf84fc117e34":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",
    "80bf6cde56de0f73954554093e68c98708a141021797b76f3217a81ba0072173":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-musllinux_1_2_x86_64.whl",
    "256537c2c1083a39901c4ea3ba509eed0617fc194258186c647a67ef554d3b2b":
        "qrcode_barcode_scanner-0.11.0-cp38-abi3-win_amd64.whl",
}

installed = importlib.metadata.version("qrcode-barcode-scanner")
assert installed == VERSION, f"version {installed} is installed, this notebook expects {VERSION}"
module = next(path for path in pathlib.Path(scanner.__file__).parent.iterdir()
              if path.suffix in (".so", ".pyd"))
digest = hashlib.sha256(module.read_bytes()).hexdigest()
wheel = PUBLISHED_MODULES.get(digest)

print("Python", platform.python_version(), "on", platform.system(), platform.machine())
print("qrcode-barcode-scanner", installed)
print("Native module SHA-256:", digest)
assert wheel is not None, "this native module is not one of the modules published on PyPI"
print("It is the module of the PyPI wheel", wheel)

## 3. Get the sample image

The sample holds one QR code and one Code 128 barcode. It comes from the scanner's repository, at
the release tag, and the cell checks its SHA-256 before using it.

In [ ]:
import urllib.request

from IPython.display import Image, display

SAMPLE_URL = ("https://raw.githubusercontent.com/qrcode-ai-com/qrcode-barcode-scanner/"
              "v0.11.0/fixtures/symbology/mixed-code128-qr.png")
SAMPLE_SHA256 = "c636e8d56df483ff1ec06dbe9b5bcba226c579f8fc1c3d10c4ccf4e100762ec8"

with urllib.request.urlopen(SAMPLE_URL, timeout=60) as response:
    sample = response.read()
assert hashlib.sha256(sample).hexdigest() == SAMPLE_SHA256, "the sample is not the expected file"
display(Image(data=sample, width=480))

## 4. Scan it with the three profiles

`full` runs every stage and scores how well a QR code scans. `fast` skips the deep QR stages.
`frame` only decodes, as for camera frames. `budget_ms=0` removes the time budget, so every scan
runs to its end and `trace.budget_exhausted` is false.

In [ ]:
EXPECTED = {("qr_code", "https://qrc-ai.com/76xMa"), ("code128", "STOCK-4827")}

reports = {}
for profile in ("fast", "frame", "full"):
    report = scanner.scan(sample, profile=profile, budget_ms=0)
    reports[profile] = report
    found = [(row["symbology"], row["content"]["text"]) for row in report["detections"]]
    assert len(found) == 2 and set(found) == EXPECTED, (profile, found)
    assert report["trace"]["budget_exhausted"] is False, profile
    print(f"{profile:<5}", " · ".join(f"{symbology}: {text}" for symbology, text in sorted(found)))

score = reports["full"]["score"]
print("Scannability of the first code:", score["value"], "out of 100,", score["grade"])

## 5. Read a report

A report is a `dict`. Each detection gives the symbology, the decoded content, the kind of payload
and where each physical copy of the code was measured in the image.

In [ ]:
for row in reports["full"]["detections"]:
    print(row["symbology"])
    print("  text       ", row["content"]["text"])
    print("  payload    ", row["payload"]["kind"])
    for occurrence in row["occurrences"]:
        geometry = occurrence["geometry"]
        points = ", ".join(f"({point['x']:.0f}, {point['y']:.0f})" for point in geometry["points"])
        print("  measured as", geometry["kind"], "at", points)

## 6. An image without a code, and a file that is not an image

An image with no code gives an empty list: a valid result, not an error. Bytes that are not an
image raise `ValueError` with the code `QRS-001`.

In [ ]:
import struct
import zlib


def white_png(width=64, height=64):
    """A plain white PNG, built here so that the cell needs no other package."""
    def chunk(kind, data):
        body = kind + data
        return struct.pack(">I", len(data)) + body + struct.pack(">I", zlib.crc32(body))
    row = b"\x00" + b"\xff" * (3 * width)
    return (b"\x89PNG\r\n\x1a\n"
            + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(row * height)) + chunk(b"IEND", b""))


empty = scanner.scan(white_png(), profile="full", budget_ms=0)
assert empty["detections"] == [] and empty["trace"]["budget_exhausted"] is False
print("White image: no code found, and the scan finished.")

try:
    scanner.scan(b"not an image", profile="full", budget_ms=0)
except ValueError as error:
    assert "QRS-001" in str(error), error
    print("Not an image:", error)
else:
    raise AssertionError("bytes that are not an image were accepted")

## 7. Scan your own image

Upload an image to this session, put its path below and run the cell. A decoded link is printed as
text: the scanner never opens it.

In [ ]:
MY_IMAGE = None  # for example "label.png"

if MY_IMAGE:
    report = scanner.scan(pathlib.Path(MY_IMAGE).read_bytes(), profile="full", budget_ms=0)
    for row in report["detections"]:
        print(row["symbology"], "·", row["content"]["text"])
    if not report["detections"]:
        print("No code found.")
else:
    print("Set MY_IMAGE to the path of an image, then run this cell again.")

## Go further

- [Python API](https://github.com/qrcode-ai-com/qrcode-barcode-scanner/blob/v0.11.0/docs/api/python.mdx)
- [The report, field by field](https://github.com/qrcode-ai-com/qrcode-barcode-scanner/tree/v0.11.0/spec)
- [How the score is computed](https://github.com/qrcode-ai-com/qrcode-barcode-scanner/blob/v0.11.0/docs/SCORING.md)

To remove the package from this session, run `%pip uninstall --yes qrcode-barcode-scanner`.